# 1.- S03 | Practica guiada: del analisis ad-hoc al dataset preparado

**Master en Analitica de Negocios — UNIE · Ciencia de Datos Aplicada a Negocio · Sesion 3 (2026-05-18)**

Docente: Jaime Pineda · Caso ancla: *No-Show Appointments* (ausentismo en citas medicas)

En **S02** abrimos el dataset a escala: 110.527 citas, una fila = una cita, baseline 20,2 %, y construimos `NoShow`, `WaitingDays` y `AppointmentWeekday`. Hoy, **S03**, estamos en la fase **Data Preparation** de CRISP-DM: convertimos esas primeras evidencias en un **dataset preparado y reproducible** y en **tablas que un stakeholder pueda leer**.

## 1.1.- Como usar este notebook

- Ejecuta en orden.
- Local: usa `data_path`. Colab: sustituye solo la celda de carga por Drive.
- Lee cada output antes de avanzar: **codigo -> resultado -> decision**.
- `# MICROEJERCICIO` y `# RETO OPCIONAL` no rompen el pipeline.
- Hoy no hay graficos: salida = tablas + `noshow_prepared.csv`.

## 1.2.- Objetivos de la sesion

Al cerrar S03 debes poder:

1. Limpiar y validar el dataset con criterios `DAMA-5`.
2. Encapsular preparacion en `prepare_basic(df)` -> `prepare(df)`.
3. Crear `AgeGroup`, `WaitingBin`, `LongWait`, `IsWeekend` sin numpy.
4. Resumir segmentos con tasa + denominador.
5. Enriquecer por barrio con `merge`, `min_n` y `Hotspot = 0.22`.
6. Exportar `noshow_prepared.csv` y actualizar Canvas v3.

> Criterio de exito: un CSV reproducible que puedas explicar.

# 2.- B0 · Apertura: de S02 a Data Preparation

> Lente principal: `CRISP-DM: Data Preparation`. Sin codigo. Recap de S02, ubicacion en el curso, las cuatro lentes de hoy y el mapa de la sesion.

## 2.1.- Que traemos de S02

| Punto | Evidencia |
|---|---|
| Datos validos | **110.521** citas de **110.527** brutas; se retiran 6 filas invalidas |
| Variables creadas | `NoShow`, `WaitingDays`, `AppointmentWeekday` |
| Baseline | no-show **20,2 %** |
| Hallazgo fuerte | espera `>14` dias: **32,7 %** vs **16,0 %** |
| Pendiente | comparar cita vs paciente; cerrar Canvas v3 |

Hoy convertimos el analisis de S02 en preparacion reproducible.

## 2.2.- CRISP-DM: posicion de S03

| Fase | En el curso |
|---|---|
| Business Understanding | S01: problema, KPI, Canvas |
| Data Understanding | S02/S04: EDA, calidad, visualizacion |
| **Data Preparation** | **S03: limpieza, variables, pipeline, CSV preparado** |
| Modeling | S05-S10: modelos y metricas |
| Deployment | S11: entrega y reproducibilidad |

Hoy: **Data Preparation**. S04 visualiza; S05+ modela.

## 2.3.- Lentes de trabajo

| Lente | Uso en S03 |
|---|---|
| `CRISP-DM` | ubicar la preparacion dentro del proyecto |
| `DAMA-5` | justificar limpieza, grano y muestra minima |
| `EDA` | crear variables que faltan para preguntar mejor |
| `PECO` | leer tablas por segmento como comparaciones |
| `Metodos Pandas` | ejecutar cada decision con codigo trazable |

## 2.4.- Mapa de la sesion S03

| Bloque | Verbo | Producto |
|---|---|---|
| B1 | limpiar | nombres, nulos, duplicados, grano |
| B2 | encapsular | `prepare_basic(df)` |
| B3-B4 | derivar | `AgeGroup`, `WaitingBin`, `LongWait`, `IsWeekend`; `prepare(df)` |
| B5-B6 | resumir | tablas por segmento y cruces |
| B7 | enriquecer | tasa por barrio + `Hotspot` |
| B8 | exportar | `noshow_prepared.csv` + Canvas v3 |

# 3.- B1 · Limpieza sistematica

> Lente principal: `DAMA-5: completitud + unicidad + consistencia`. Primero arreglamos los nombres mal escritos, luego comprobamos nulos y duplicados, y por ultimo entendemos el **grano** de la tabla.

## 3.1.- Carga del dataset

En VS Code/local ejecuta la celda siguiente. En Google Colab, si el CSV esta en Drive, sustituye la carga local por este bloque:

```python
# Metodo Google Colab
import pandas as pd
from google.colab import drive

# Montar Drive y cargar dataset
drive.mount('/content/drive')
data_path = "/content/drive/MyDrive/Classroom/KaggleV2-May-2016.csv"
df = pd.read_csv(data_path)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head(3)
```

La variable importante es `data_path`: el resto del notebook reutiliza esa ruta para reconstruir el dataset desde cero.

In [ ]:
# Load the No-Show Appointments dataset.
import pandas as pd

data_path = "../../../../Recursos/Datasets/noshowappointments/KaggleV2-May-2016.csv"
df = pd.read_csv(data_path)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head(3)

## 3.2.- Ficha · `Metodos Pandas: limpieza sistematica`

| Metodo | Para que sirve | Valor analitico aqui |
|---|---|---|
| `df.rename(columns={...})` | renombrar columnas | arreglar `Hipertension`/`Handcap` mal escritas |
| `df.isna().sum()` | contar nulos por columna | comprobar **completitud** |
| `df.dropna()` | eliminar filas con nulos | aplicar el patron (aunque hoy no pierda nada) |
| `df.duplicated().sum()` | contar filas duplicadas | comprobar **unicidad de fila** |
| `df["col"].nunique()` | valores distintos en una columna | comparar pacientes unicos vs numero de citas → **grano** |

**Validacion minima:** despues de cada limpieza, comprobar **cuantas filas quedan** y **por que**. Una limpieza que no sabes justificar es ruido, no preparacion.

In [ ]:
# Renombrar las dos columnas con erratas del dataset original.
# Decidimos los nombres correctos en ingles para mantener la convencion del curso.
df = df.rename(columns={"Hipertension": "Hypertension", "Handcap": "Handicap"})
df.columns

**Lectura — `DAMA-5: consistencia` (nombres) y `validez` (`Handicap`):** las dos columnas estaban mal escritas en la fuente. Ademas, `Handicap` no es binaria: toma valores 0, 1, 2, 3, 4 (grado de discapacidad). Hoy la dejamos como **variable ordinal**; si manana la usaramos como "tiene/no tiene", habria que binarizarla y dejarlo documentado.

In [ ]:
# DAMA-5: completitud. Comprobamos nulos antes de asumir nada.
df.isna().sum()

In [ ]:
# Aplicamos el patron de limpieza de nulos. En este dataset no se pierde ninguna fila,
# pero el patron es el mismo siempre: comprobar -> decidir -> registrar.
filas_antes = len(df)
df_sin_nulos = df.dropna()
print(f"Filas antes: {filas_antes:,} | despues de dropna(): {len(df_sin_nulos):,} | perdidas: {filas_antes - len(df_sin_nulos)}")

**Lectura — `DAMA-5: completitud ✓`:** el dataset No-Show no tiene nulos, asi que `dropna()` no retira ninguna fila. **Aun asi se comprueba**: en un caso real (tu TFM, por ejemplo) lo normal es que si haya huecos, y la decision —eliminar si afectan a <5 % de filas, imputar con logica de negocio si no— hay que tomarla con datos delante, no por costumbre. Como no perdemos nada, seguimos trabajando sobre `df`.

In [ ]:
# DAMA-5: unicidad de fila. Hay filas exactamente repetidas? Y citas (AppointmentID) repetidas?
print("Filas duplicadas (todas las columnas):", df.duplicated().sum())
print("AppointmentID duplicados:", df.duplicated(subset=["AppointmentID"]).sum())

**Lectura — `DAMA-5: unicidad de fila ✓`:** cero duplicados a nivel de fila y cero `AppointmentID` repetidos. La tabla esta limpia en esa dimension. Pero "no hay filas duplicadas" no es lo mismo que "no hay ambiguedad de granularidad" — eso lo miramos ahora.

In [ ]:
# La unicidad que SI importa aqui: cual es el grano de la tabla?
# Una fila es una cita. Cuantos pacientes distintos hay detras de esas citas?
n_appointments = len(df)
n_patients = df["PatientId"].nunique()
appointments_per_patient = df["PatientId"].value_counts()
pct_single_appointment = (appointments_per_patient == 1).mean()

print(
    f"Citas={n_appointments:,} | Pacientes={n_patients:,} | "
    f"mediana citas/paciente={appointments_per_patient.median():.0f} | "
    f"max={appointments_per_patient.max()} | una cita={pct_single_appointment:.1%}"
)

**Lectura — `DAMA-5: grano`:** 110.527 citas, 62.299 pacientes. La tabla no esta duplicada; su grano es **cita**. Una lectura por cita no responde lo mismo que una lectura por paciente. Lo retomamos en B5.

# 4.- B2 · De celdas sueltas a una funcion: `prepare_basic(df)`

> Lente principal: `CRISP-DM: Data Preparation reproducible`. En S02 hicimos a mano: parsear fechas, normalizar, calcular `WaitingDays`, construir `NoShow`, crear `AppointmentWeekday` y filtrar filas invalidas. Hoy distinguimos lo que fue **analisis** de lo que fue **transformacion**, y metemos solo las transformaciones —mas el `rename`— en **una funcion propia**. Es nuestro primer `def`.

## 4.1.- Ficha · que es una funcion

| Elemento | Idea |
|---|---|
| Que es | un **nombre que guarda un procedimiento**: recibe parametros, hace pasos, y **devuelve** un resultado (`return`) |
| Para que sirve | dejar de copiar-pegar celdas; aplicar el mismo proceso a datos distintos; documentar "como se preparan estos datos" en un sitio |
| Valor analitico aqui | un solo `prepare_basic(df)` reproduce **toda** la preparacion de S02 — y manana podras reusarlo o ampliarlo |
| Validacion | la salida debe tener las filas que esperabamos: 110.521 (las validas) + las columnas nuevas de S02 |

> Una funcion es la forma de no copiar-pegar. No es programacion avanzada: es lo que hace **reproducible** la preparacion.

## 4.2.- Anatomia de una funcion: recibe, transforma, devuelve

Antes de tocar `df`, un ejemplo neutro con el mismo esqueleto: un parametro entra, se reasigna paso a paso, y `return` lo devuelve. Manana `texto` sera `df` y los pasos seran los de S02.

| Parte | Que es |
|---|---|
| `parametro` (`texto`) | lo que **entra** (el dato crudo) |
| cuerpo | los **pasos** que transforman, reasignando la misma variable |
| `return` | lo que **sale** (el dato ya preparado) |

In [ ]:
# Primer def del curso. Esqueleto identico al de prepare_basic: recibe -> pasos -> return.
def preparar_nombre(texto):
    texto = texto.strip()      # paso 1
    texto = texto.upper()      # paso 2
    return texto               # lo que la funcion devuelve

preparar_nombre("  ana  ")

**Lectura:** la funcion no "imprime" — **devuelve**. El `return texto` es lo que la funcion da para que lo uses (aqui, `"ANA"`). Ese es exactamente el patron de `prepare_basic`: entra el crudo, se transforma, sale el preparado.

## 4.3.- Que de S02 va en la funcion y que se queda

| En S02... | Que era | Se reproduce hoy? |
|---|---|---|
| **Analisis** — mirar para entender | `shape`/`head`/`info`; `value_counts` + baseline 20,2 %; anomalias (`Age < 0`); H1/H2/H3 | No: fue **Data Understanding**, se queda en S02 |
| **Transformacion** — cambiar la tabla | crear `NoShow`, `WaitingDays`, `AppointmentWeekday`; filtrar filas invalidas | **Si: es Data Preparation → va en la funcion** |

> Regla: una funcion reproducible guarda lo que **cambia** los datos, no lo que solo los **mira**. Por eso `prepare_basic` lleva solo las transformaciones de S02 (mas el `rename` de B1).

## 4.4.- Transformaciones S02, condensadas

No se ejecuta aqui. Es el bloque que vamos a envolver en una funcion.

```python
# objetivo legible
df["NoShow"] = (df["No-show"] == "Yes").astype(int)

# fechas -> variables
sd = pd.to_datetime(df["ScheduledDay"]).dt.normalize()
ad = pd.to_datetime(df["AppointmentDay"]).dt.normalize()
df["WaitingDays"] = (ad - sd).dt.days
df["AppointmentWeekday"] = ad.dt.day_name()

# validacion minima
valid = (df["Age"] >= 0) & (df["WaitingDays"] >= 0)
df = df.loc[valid].copy()   # 110.527 -> 110.521
```

## 4.5.- `prepare_basic(df)`: ese bloque, envuelto

Cogemos el bloque de arriba, le anteponemos el `rename` de B1 y lo envolvemos en `def ... return` — igual que `preparar_nombre`.

In [ ]:
# Encapsulamos en una funcion todo lo que S02 hizo a mano + el rename de B1.
# Convencion: la funcion recibe el DataFrame crudo y devuelve una copia limpia.
def prepare_basic(df):
    """Reproduce la preparacion basica de S02: nombres, NoShow, fechas,
    WaitingDays, AppointmentWeekday y filtro de filas validas."""
    df = df.copy()
    # 1) nombres correctos (rename de B1)
    df = df.rename(columns={"Hipertension": "Hypertension", "Handcap": "Handicap"})

    # 2) objetivo legible 0/1 (S02-B3)
    df["NoShow"] = (df["No-show"] == "Yes").astype(int)

    # 3) fechas normalizadas (sin hora) y variables derivadas (S02-B5)
    sd = pd.to_datetime(df["ScheduledDay"]).dt.normalize()
    ad = pd.to_datetime(df["AppointmentDay"]).dt.normalize()
    df["WaitingDays"] = (ad - sd).dt.days
    df["AppointmentWeekday"] = ad.dt.day_name()

    # 4) filtro de filas claramente invalidas (S02-B6)
    valid = (df["Age"] >= 0) & (df["WaitingDays"] >= 0)
    
    return df.loc[valid].copy()

In [ ]:
# Ejecutamos la funcion sobre el CSV crudo y comparamos con lo que esperabamos de S02.
df = prepare_basic(pd.read_csv(data_path))
print("Forma tras prepare_basic:", df.shape)              # esperado: (110521, 17)
print("Tasa de no-show:", round(df["NoShow"].mean(), 3)) # esperado: 0.202
df.head(3)

**Lectura:** `prepare_basic` reproduce en **una linea** lo que en S02 fueron muchas celdas: misma cifra de 110.521 filas validas, mismo baseline 20,2 %. El `df.copy()` de entrada y el `return df.loc[...].copy()` de salida evitan el `SettingWithCopyWarning`. A partir de aqui, `df` siempre viene de la funcion — si necesitamos rehacer la preparacion, basta volver a llamarla.

## 4.6.- Prompt clinic 3 · refactorizar a funcion

**Prompt debil:**
> `Convierte mis celdas en una funcion.`

**Prompt util:**
> Tengo celdas Pandas que parsean `ScheduledDay`/`AppointmentDay`, calculan `WaitingDays`, crean `NoShow` y `AppointmentWeekday`, y filtran `Age<0` / `WaitingDays<0`. Conviertelas en `def prepare_basic(df): ... return df_limpio`, trabajando sobre una copia y validando que devuelve 110.521 filas.

**Por que funciona:** fija entrada, transformaciones, salida esperada y validacion.

> **Microreto (90 s):** adapta el prompt a tu TFM.

# 5.- B3 · Variables derivadas I: binning con `pd.cut`

> Lente principal: `EDA: variables que faltan` + `Metodos Pandas: binning`. La edad y la espera son numeros continuos; para **comparar tasas por grupo** nos conviene agruparlos en tramos con sentido de negocio.

## 5.1.- Ficha · `pd.cut`

| Metodo | Uso | Check |
|---|---|---|
| `pd.cut(serie, bins, labels)` | convertir continuo -> tramos | sin `NaN` fuera de rango |
| `bins` / `labels` | cortes y nombres | n etiquetas = n cortes - 1 |
| `value_counts(sort=False)` | volumen por tramo | ningun tramo vacio o absurdo |

**Idea:** los tramos convierten edad/espera en palancas de negocio. El corte se justifica, no se deja al azar.

In [ ]:
# AgeGroup: etapas vitales. Los cortes responden a una logica de negocio (no son arbitrarios):
# 0-12 ninez, 13-17 adolescencia, 18-35 joven, 36-64 adulto, 65+ senior.
# pd.cut es right-inclusive: (-1, 12] = 0 a 12 anios (ya filtramos Age >= 0 en prepare_basic).
age_bins = [-1, 12, 17, 35, 64, 200]
age_labels = ["Ninez", "Adolescencia", "Joven", "Adulto", "Senior"]
df["AgeGroup"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels)
df["AgeGroup"].value_counts(sort=False)

In [ ]:
# WaitingBin: tramos de espera. El primer corte (0-3 dias) separa "casi inmediato"
# del resto, que en S02 ya apuntaba a mas no-show.
wait_bins = [-1, 3, 7, 14, 30, 10_000]
wait_labels = ["0-3", "4-7", "8-14", "15-30", "30+"]
df["WaitingBin"] = pd.cut(df["WaitingDays"], bins=wait_bins, labels=wait_labels)
df["WaitingBin"].value_counts(sort=False)

**Lectura:** la mayoria de citas (≈53 mil) cae en el tramo **0-3 dias** — esperas cortas. `Adulto` es el grupo de edad mayor (≈43 mil citas). Los tramos largos tienen menos volumen pero, como veremos en B5-B6, **mas no-show**. Ningun tramo queda vacio ni absorbe todo, asi que el corte es utilizable.

## 5.2.- Microejercicio 1 · tu propio binning

Define un binning **distinto** para `WaitingDays` (o para `Age`) y mira como cambia el reparto. Idea de negocio sugerida: `mismo dia` · `esta semana` · `este mes` · `mas de un mes`. Justifica el corte con una frase de negocio. Guarda el resultado en una variable suelta (no en `df`).

In [ ]:
# MICROEJERCICIO 1 — Tu codigo aqui (intentalo antes de mirar la solucion sugerida).
# Crea una variable suelta (no en df) con tus propios cortes y haz value_counts(sort=False).

In [ ]:
# MICROEJERCICIO 1 — solucion sugerida (una de muchas posibles).
# Idea de negocio: "mismo dia" se comporta distinto del resto.
# Lo guardamos en una variable suelta (no en df) para no arrastrarlo al dataset preparado.
mi_bins = [-1, 0, 7, 30, 10_000]
mi_labels = ["mismo dia", "esta semana", "este mes", "mas de un mes"]
waiting_bin2 = pd.cut(df["WaitingDays"], bins=mi_bins, labels=mi_labels)
waiting_bin2.value_counts(sort=False)

# 6.- B4 · Variables derivadas II: vectorizacion y `prepare(df)`

> Lente principal: `Metodos Pandas: vectorizacion`. Una columna de un DataFrame es una **Series** (lo visteis en S02). **Vectorizar** = aplicar una operacion a **toda la Series de una vez**, sin recorrerla fila a fila con un `for`. Reusamos el idioma de S02 `(condicion).astype(int)` + un unico metodo nuevo, `.isin`. Nada de numpy: numpy se introduce en S05.

## 6.1.- Ficha · vectorizacion

| Metodo | Para que sirve | Valor analitico aqui |
|---|---|---|
| `(condicion).astype(int)` | mascara booleana → 0/1 (ya visto en S02; asi nacio `NoShow`) | `LongWait = 1` si `WaitingDays > 14` |
| `serie.isin([lista])` | ¿el valor esta en la lista? → mascara booleana | `IsWeekend` desde `AppointmentWeekday` |

**Anti-patron:** un `for` que recorra `df.iterrows()` para rellenar una columna fila a fila — mas lento y mas dificil de leer. `.isin` devuelve una mascara como `>` o `==`; por eso encadena con `.astype(int)` exactamente igual que `NoShow` en S02. numpy reaparecera en S05 (ML), donde gana su sitio.

## 6.2.- El patron de S02 + un metodo nuevo: `.isin`

```text
Ya lo sabeis (S02):  (condicion) -> mascara True/False ;  .astype(int) -> 0/1
   asi nacio NoShow:  (df["No-show"] == "Yes").astype(int)

Nuevo, 1 metodo:     serie.isin([lista])  -> ¿el valor esta en la lista?
```

`.isin` devuelve una mascara booleana — igual que `>` o `==` —, por eso encadena con `.astype(int)`:

In [ ]:
# .isin: micro-ejemplo antes -> despues (la salida es una mascara, como > o ==).
dias = pd.Series(["Monday", "Saturday", "Sunday", "Tuesday"])
mask = dias.isin(["Saturday", "Sunday"])
mask.astype(int)

In [ ]:
# LongWait: flag 0/1 con el idioma de S02 (condicion).astype(int).
df["LongWait"] = (df["WaitingDays"] > 14).astype(int)
short_wait_rate = df.loc[df["LongWait"] == 0, "NoShow"].mean()
long_wait_rate = df.loc[df["LongWait"] == 1, "NoShow"].mean()

print(df["LongWait"].value_counts())
print(f"No-show espera <=14 dias: {short_wait_rate:.3f}")
print(f"No-show espera >14 dias: {long_wait_rate:.3f}")

In [ ]:
# IsWeekend: el unico metodo nuevo de hoy, .isin, encadenado con .astype(int).
df["IsWeekend"] = df["AppointmentWeekday"].isin(["Saturday", "Sunday"]).astype(int)
df["IsWeekend"].value_counts()

**Lectura — validez:** `LongWait` separa riesgo (16,0 % vs 32,7 %). `IsWeekend` existe, pero con **n=39** no sostiene una decision.

## 6.3.- `prepare(df)`: una funcion que crece

`prepare` = `prepare_basic` + las cuatro variables derivadas de hoy. Una funcion que **llama a otra** y crece por etapas: si manana cambia una regla, se cambia en un sitio.

In [ ]:
# Ampliamos prepare_basic a prepare(df): una funcion que crece por etapas.
def prepare(df):
    """Preparacion completa de S03: prepare_basic + AgeGroup, WaitingBin, LongWait, IsWeekend."""
    df = prepare_basic(df)
    df["AgeGroup"] = pd.cut(df["Age"], bins=[-1, 12, 17, 35, 64, 200],
                            labels=["Ninez", "Adolescencia", "Joven", "Adulto", "Senior"])
    df["WaitingBin"] = pd.cut(df["WaitingDays"], bins=[-1, 3, 7, 14, 30, 10_000],
                              labels=["0-3", "4-7", "8-14", "15-30", "30+"])
    df["LongWait"] = (df["WaitingDays"] > 14).astype(int)
    df["IsWeekend"] = df["AppointmentWeekday"].isin(["Saturday", "Sunday"]).astype(int)
    return df

In [ ]:
# Reconstruimos df desde la funcion. Mismo dataset que veniamos armando a mano,
# pero ahora en una sola llamada. De aqui en adelante df viene de prepare().
df = prepare(pd.read_csv(data_path))
print("Forma tras prepare:", df.shape)   # esperado: (110521, 21)
df.head(3)

**Lectura:** `prepare(df)` es la **definicion ejecutable** de "como se preparan estos datos". Si manana cambia una regla (otro corte de edad, otro umbral de espera), se cambia **en un sitio** y todo lo que viene detras se recalcula. Eso es Data Preparation reproducible.

# 7.- Descanso (20:00-20:20)

A la vuelta: abrimos el `groupby` de S02 "por dentro" con `agg`, montamos tablas dinamicas y enriquecemos el dataset con una metrica calculada por barrio.

# 8.- B5 · Agregaciones potentes: `groupby + agg`

> Lente principal: `PECO: P/E/C/O` + `Metodos Pandas: resumen por grupos`. Un `groupby` es una **comparacion**: poblacion (las citas de cada grupo), exposicion (el grupo), comparacion (los demas grupos), outcome (`NoShow`). Con `agg` pedimos varias metricas de golpe — y siempre **tasa y conteo juntos**.

## 8.1.- Ficha · `groupby + agg`

| Metodo | Uso | Check |
|---|---|---|
| `df.groupby("col")` | separar por segmento | grano correcto |
| `.agg(nombre=("col", "funcion"))` | varias metricas con nombre | tasa + n juntos |
| `"mean"`, `"size"`, `"median"` | tasa, denominador, mediana | no leer tasas sin volumen |
| `.sort_values("col")` | priorizar grupos | arriba no siempre = accion |
| `observed=True` | ocultar categorias vacias | util con `Categorical` |

**Regla:** tasa sin `n` no se interpreta.

In [ ]:
# Tasa de no-show por grupo de edad, con conteo y mediana de espera.
# observed=True porque AgeGroup es Categorical (no queremos categorias vacias).
age_summary = df.groupby("AgeGroup", observed=True).agg(
    noshow_rate=("NoShow", "mean"),       # media de 0/1 = tasa
    citas=("NoShow", "size"),             # denominador
    espera_mediana=("WaitingDays", "median"),
)
age_summary = age_summary.sort_values("noshow_rate", ascending=False)
age_summary.round(3)

**Lectura — negocio:** la tasa de no-show **no crece con la edad**. El pico esta en **Adolescencia (≈26,6 %)** y el minimo en **Senior (≈15,5 %)**; los `Adulto` (el grupo mas grande, ≈43 mil citas) estan por debajo del baseline 20,2 %. La mediana de espera es parecida entre grupos (3-4 dias), asi que la diferencia de tasa no se explica solo por "los jovenes esperan mas". Esto es una **hipotesis para S04**: ¿el efecto edad sobrevive cuando controlamos por espera? Lo cruzaremos en B6.

In [ ]:
# Cita vs paciente: el grano cambia la respuesta (era la pregunta abierta de S02).
# Cuantas citas tiene cada paciente?
appts_per_patient = df.groupby("PatientId").size()
appts_per_patient.describe()

In [ ]:
# Asisten distinto los pacientes recurrentes? Comparamos la tasa de no-show
# de los que tienen >=3 citas frente a los que tienen 1 sola. .isin: ¿es recurrente?
recurrent_ids = appts_per_patient[appts_per_patient >= 3].index
single_ids = appts_per_patient[appts_per_patient == 1].index
rate_recurrent = df.loc[df["PatientId"].isin(recurrent_ids), "NoShow"].mean()
rate_single = df.loc[df["PatientId"].isin(single_ids), "NoShow"].mean()
n_recurrent = df["PatientId"].isin(recurrent_ids).sum()
print(f"Pacientes con >=3 citas -> tasa no-show: {rate_recurrent:.3f}  (n citas = {n_recurrent:,})")
print(f"Pacientes con 1 sola cita -> tasa no-show: {rate_single:.3f}")

**Lectura — grano:** 62.298 pacientes; mediana 1 cita, max 88. Pacientes con >=3 citas: **21,1 %** no-show; pacientes con 1 cita: **18,8 %**. Diferencia modesta (**+2,3 p.p.**), pero suficiente para recordar que cita y paciente no son la misma unidad analitica.

## 8.2.- Microejercicio 2 · tu propio `groupby + agg`

Elige un segmento relevante (`Scholarship`, `SMS_received`, `Hypertension`, `Gender`...) y resume con `agg`: la **tasa de no-show**, el **numero de citas** y una **tercera metrica**. Ordena por tasa y escribe una frase de lectura. (Ojo si eliges `SMS_received`: tienen MAS no-show → probable sesgo de seleccion, no causalidad.)

In [ ]:
# MICROEJERCICIO 2 — Tu codigo aqui (intentalo antes de mirar la solucion sugerida).

In [ ]:
# MICROEJERCICIO 2 — solucion sugerida (con SMS_received; vale cualquier segmento razonable).
sms_summary = df.groupby("SMS_received").agg(
    noshow_rate=("NoShow", "mean"),
    citas=("NoShow", "size"),
    espera_mediana=("WaitingDays", "median"),
)
sms_summary = sms_summary.sort_values("noshow_rate", ascending=False)
print(sms_summary.round(3))
# Lectura: los que reciben SMS (1) tienen MAS no-show (0,276) que los que no (0,167).
# Casi seguro sesgo de seleccion: el SMS se manda justo a citas con mas espera.
# Asociacion observacional != causalidad.

# 9.- B6 · Tablas dinamicas: `pivot_table` y `crosstab`

> Lente principal: `Metodos Pandas: tablas dinamicas`. Cuando queremos cruzar **dos** variables, una tabla dinamica es un "heatmap mental" antes del visual de S04: filas = una variable, columnas = otra, celdas = una metrica.

## 9.1.- Ficha · `pivot_table` y `crosstab`

| Metodo | Responde | Check |
|---|---|---|
| `pd.pivot_table(...)` | tasa en la cruz de dos variables | comparar con marginales |
| `margins=True` | totales fila/columna | baseline visible |
| `pd.crosstab(...)` | volumen por combinacion | evitar celdas fragiles |
| `normalize="index"` | % por fila | cada fila suma 100 % |

**Regla:** tasa alta + poco volumen = cautela, no prioridad.

In [ ]:
# Tasa de no-show por la cruz AgeGroup x WaitingBin. aggfunc="mean" sobre NoShow (0/1) = tasa.
# margins=True añade la fila/columna All (los marginales de B5).
rate_pivot = pd.pivot_table(
    df, values="NoShow", index="AgeGroup", columns="WaitingBin",
    aggfunc="mean", observed=True, margins=True,
)
rate_pivot.round(3)

**Lectura — negocio:** la **espera domina sobre la edad**. Dentro de cualquier fila de edad, la tasa sube de izquierda (0-3 d, ~9 %) a derecha (30+ d, 30-40 %). La celda mas alta es **Adolescencia · 30+ ≈ 42,5 %**; la mas baja, **Senior · 0-3 ≈ 8,7 %**. La fila `All` reproduce los marginales de B5 (baseline 20,2 %). Conclusion candidata: **la espera es la palanca; la edad modula**.

In [ ]:
# Reparto: que % de las citas de cada grupo de edad cae en cada tramo de espera?
# normalize="index" -> cada fila suma 1; lo mostramos en % por fila (x100) para leerlo sin friccion.
vol_crosstab = pd.crosstab(df["AgeGroup"], df["WaitingBin"], normalize="index")
(vol_crosstab * 100).round(1)

**Lectura — negocio:** leer una fila: el **51** de Niñez = el 51 % de sus citas son espera 0-3 d. El reparto por tramo de espera es **parecido entre edades** (~50 % en 0-3 para todos los grupos). Por eso, cuando el `pivot_table` muestra mas no-show en jovenes, ese exceso **no** se debe a que esperen mas — hay algo propio de la edad. Otra hipotesis para S04. Recordatorio operativo: tasa alta + poco volumen no es donde poner los recursos.

# 10.- B7 · Enriquecimiento por barrio: `merge`

> Lente principal: `Metodos Pandas: merge` + `DAMA-5: validez (muestra minima)`. Calculamos la tasa de no-show **por barrio** y la **reincorporamos** a cada cita con un `merge`. Pero antes hay que decidir que hacer con los barrios de muy pocas citas — y nombrar un riesgo metodologico.

## 10.1.- Ficha · `Metodos Pandas: merge`

| Metodo | Para que sirve | Valor analitico aqui |
|---|---|---|
| `tabla_a.merge(tabla_b, on="clave", how="left")` | combinar dos tablas por una **clave comun** | añadir a cada cita la tasa de su barrio |
| `how="left"` | conservar **todas** las filas de la izquierda | no perder ninguna cita |
| `.reset_index()` | pasar un `groupby` de indice a columna | dejar `Neighbourhood` como columna para el `merge` |

**Validacion minima:** despues de un `merge`, comprobar que el numero de filas **no cambia** (con `how="left"` y clave unica en la derecha, debe quedar igual). Si crece, la clave estaba duplicada en la derecha; si baja, perdiste filas.

## 10.2.- `merge`: lo importante es `how`

`how` decide que filas sobreviven al unir tablas.

```text
left   -> conserva todas las citas                 (hoy)
inner  -> solo claves presentes en ambas tablas
right  -> conserva toda la tabla derecha
outer  -> conserva todo, case o no
```

Hoy usamos `left`: la tabla principal son las citas; la tasa de barrio solo añade columnas. Check esperado: mismas filas y 0 nulos en `NoShowRate_by_Neigh`.

In [ ]:
# Tasa de no-show por barrio + numero de citas (siempre los dos juntos).
# reset_index para que Neighbourhood vuelva a ser columna y podamos hacer el merge.
neigh_rate = df.groupby("Neighbourhood").agg(
    NoShowRate_by_Neigh=("NoShow", "mean"),
    NCitas_by_Neigh=("NoShow", "size"),
).reset_index()
print("Barrios distintos:", len(neigh_rate))
neigh_rate.sort_values("NoShowRate_by_Neigh", ascending=False).head()

**Lectura — `DAMA-5: validez (muestra minima)`:** el barrio con la tasa mas alta (`ILHAS OCEÂNICAS DE TRINDADE`) tiene **tasa 100 %... con 2 citas**. Eso no es un "punto caliente": es **ruido**. Una tasa solo es informativa si hay suficientes casos detras. Necesitamos un **umbral**.

In [ ]:
# Fijamos un umbral de muestra minima y miramos los barrios "altos" de verdad.
min_n = 50
barrios_fiables = neigh_rate[neigh_rate["NCitas_by_Neigh"] >= min_n]
n_descartados = len(neigh_rate) - len(barrios_fiables)
print(f"Barrios con < {min_n} citas (poco fiables): {n_descartados}")
barrios_fiables.sort_values("NoShowRate_by_Neigh", ascending=False).head()

**Lectura:** con `min_n = 50` se quedan fuera 5 barrios; los demas son comparables. Los barrios con mas no-show "de verdad" rondan el **27-29 %** (`SANTOS DUMONT`, `SANTA CECÍLIA`, `SANTA CLARA`, `ITARARÉ`, `JESUS DE NAZARETH`) — unos 7-9 puntos por encima del baseline. **Eso si es una pista operativa.**

In [ ]:
# Reincorporamos la tasa por barrio a cada cita con un merge por la clave Neighbourhood.
# how="left": conservamos todas las citas.
n_before = len(df)
df = df.merge(neigh_rate, on="Neighbourhood", how="left")
n_after = len(df)
print(f"Filas antes del merge: {n_before:,}  | despues: {n_after:,}  | iguales? {n_before == n_after}")
df[["Neighbourhood", "NoShow", "NoShowRate_by_Neigh", "NCitas_by_Neigh"]].head()

**Lectura:** mismas filas antes y despues — el `merge` ni duplico ni perdio citas (la clave `Neighbourhood` es unica en `neigh_rate`). Ahora cada cita "sabe" la tasa de su barrio. **Cuidado:** esa tasa **incluye la propia cita** — si manana esto alimenta un modelo para predecir el no-show de esa misma cita, estariamos prediciendo la tasa del barrio con la tasa del barrio: **razonamiento circular**. En S03 la usamos como **descriptor**, no como prediccion. Se retoma en S05 (features y data leakage).

In [ ]:
# RETO OPCIONAL — flag Hotspot. Umbral de NEGOCIO fijo (no percentil): claramente sobre el baseline ~20 %.
# Puedes saltarte esta celda; la funcion de abajo ya incluye Hotspot.
HOTSPOT_MIN_RATE = 0.22   # decision de negocio, como los cortes de pd.cut en B3
df["Hotspot"] = ((df["NoShowRate_by_Neigh"] >= HOTSPOT_MIN_RATE)
                 & (df["NCitas_by_Neigh"] >= 50)).astype(int)
n_barrios_hotspot = neigh_rate.loc[
    (neigh_rate["NoShowRate_by_Neigh"] >= HOTSPOT_MIN_RATE)
    & (neigh_rate["NCitas_by_Neigh"] >= 50)
].shape[0]
print(f"Umbral de negocio: {HOTSPOT_MIN_RATE}")
print(f"Barrios marcados como hotspot: {n_barrios_hotspot}  | citas en hotspots: {df['Hotspot'].sum():,}")

**Lectura:** ≈16 barrios entran como hotspot (≈23.381 citas). El umbral es una **decision de negocio** (como los cortes de `pd.cut` en B3), no un calculo opaco; el `& NCitas >= 50` evita marcar hotspots de ruido. Mismo idioma `(condicion).astype(int)` de B4. Es un descriptor util para priorizar, pero arrastra las dos cautelas ya nombradas: **muestra minima** y **razonamiento circular**.

In [ ]:
# Empaquetamos toda la logica de barrio en una segunda funcion, que se compone con prepare().
# Parametros con valor por defecto: min_n (muestra minima) y hotspot_min_rate (umbral de negocio).
def enrich_with_neighbourhood_rate(df, min_n=50, hotspot_min_rate=0.22):
    """Añade NoShowRate_by_Neigh, NCitas_by_Neigh y Hotspot (umbral de negocio fijo)."""
    df = df.copy()
    neigh_rate = df.groupby("Neighbourhood").agg(
        NoShowRate_by_Neigh=("NoShow", "mean"),
        NCitas_by_Neigh=("NoShow", "size"),
    ).reset_index()
    df = df.merge(neigh_rate, on="Neighbourhood", how="left")
    df["Hotspot"] = ((df["NoShowRate_by_Neigh"] >= hotspot_min_rate)
                     & (df["NCitas_by_Neigh"] >= min_n)).astype(int)
    return df

In [ ]:
# Pipeline completo: dos funciones que se componen. enrich_with_neighbourhood_rate(prepare(raw)).
df = enrich_with_neighbourhood_rate(prepare(pd.read_csv(data_path)))
print("Forma final:", df.shape)   # esperado: (110521, 24)
df.head(3)

**Lectura:** el dataset preparado es ahora `enrich_with_neighbourhood_rate(prepare(raw))` — toda la preparacion en **dos llamadas componibles**. `prepare` es determinista (limpieza + derivadas); `enrich` añade un agregado calculado (de ahi sus cautelas). Esa composicion `f(g(x))` es lo que entregamos a S04.

## 10.3.- Prompt clinic 4 · validar un `merge`

**Prompt debil:**
> `Mi merge da resultados raros, que hago?`

**Prompt util:**
> `df` tiene 110.521 filas; `neigh_rate` tiene 81 barrios con `Neighbourhood` unico. Hice `df.merge(neigh_rate, on="Neighbourhood", how="left")`. Despues sigo con 110.521 filas y 0 nulos en `NoShowRate_by_Neigh`. ¿Es lo esperado? ¿Que validarias?

**Por que funciona:** aporta clave, cardinalidad y checks antes/despues.

> **Microreto (90 s):** escribe el prompt equivalente para un `merge` de tu TFM.

# 11.- B8 · Salida, Canvas v3 y puente a S04

> Lente principal: `CRISP-DM: Data Preparation documentado`. Exportamos el dataset preparado, calculamos las cifras clave para el Canvas y cerramos formulando que llevamos a S04.

## 11.1.- Que entregamos a S04

`noshow_prepared.csv` (110.521 filas × 24 columnas):

- **Identificadores y originales:** `PatientId`, `AppointmentID`, `Gender`, `Age`, `Neighbourhood`, `Scholarship`, `Hypertension`, `Diabetes`, `Alcoholism`, `Handicap`, `SMS_received`
- **Derivadas de S02:** `NoShow`, `WaitingDays`, `AppointmentWeekday`
- **Derivadas de S03:** `AgeGroup`, `WaitingBin`, `LongWait`, `IsWeekend`
- **Enriquecimiento de barrio:** `NoShowRate_by_Neigh`, `NCitas_by_Neigh`, `Hotspot`

Mas las tablas de hoy como referencia. No se entrega "a mano": S04 lo regenera con `enrich_with_neighbourhood_rate(prepare(raw))`.

In [ ]:
# Exportamos el dataset preparado. data/ esta en la carpeta de la sesion (un nivel por encima de notebooks/).
from pathlib import Path
out_dir = Path("../data") if Path("..").resolve().name == "S03_Pandas_II" else Path("data")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "noshow_prepared.csv"

df_final = enrich_with_neighbourhood_rate(prepare(pd.read_csv(data_path)))
df_final.to_csv(out_path, index=False)   # index=False: no escribir la columna de indice

print(f"Escrito: {out_path}  ({df_final.shape[0]:,} filas x {df_final.shape[1]} columnas)")

**Lectura:** `index=False` evita una primera columna `Unnamed: 0` al releer el CSV. Este fichero es **reproducible**: cualquiera con el CSV crudo y este notebook lo regenera identico. Por eso S04 no necesita que se lo "pasemos a mano" — basta volver a ejecutar `prepare()` + `enrich_with_neighbourhood_rate()`.

In [ ]:
# Cifras clave para Canvas v3.
top_barrios = neigh_rate[neigh_rate["NCitas_by_Neigh"] >= 50].sort_values(
    "NoShowRate_by_Neigh", ascending=False
).head(3)
top_neighbourhood = top_barrios.iloc[0]

print(f"Baseline no-show: {df['NoShow'].mean():.3f}")
print(f"Espera: <=14 dias {short_wait_rate:.3f} | >14 dias {long_wait_rate:.3f}")
print(f"Edad: Adolescencia {age_summary.loc['Adolescencia', 'noshow_rate']:.3f} | Senior {age_summary.loc['Senior', 'noshow_rate']:.3f}")
print(f"Top barrio fiable: {top_neighbourhood['Neighbourhood']} ({top_neighbourhood['NoShowRate_by_Neigh']:.3f})")
print(f"Hotspots: {df.loc[df['Hotspot'] == 1, 'Neighbourhood'].nunique()} barrios / {int(df['Hotspot'].sum()):,} citas")
print(f"Paciente: recurrentes >=3 {rate_recurrent:.3f} | una cita {rate_single:.3f}")

## 11.2.- Borrador editable del Canvas v3

Copia esto a tu hoja `S03_2604_canvas_update.md` y rellena. Hoy el foco es la **casilla 6** y priorizar hipotesis para S04.

### 11.2.1.- Casilla 6 — Decision candidata (v3)

```text
Decision v3:
  reforzar recordatorios en citas con WaitingDays > 14,
  priorizando barrios hotspot.

Evidencia:
  - LongWait: 16,0 % vs 32,7 %.
  - barrios fiables: top 27-29 % vs baseline 20,2 %.
  - Hotspot: umbral fijo 0,22 + min_n=50.

Limites:
  asociacion observacional; SMS sesgado; fin de semana n=39.
```

### 11.2.2.- Hipotesis priorizadas para S04

| H | Hipotesis | Evidencia S03 | Grafico S04 |
|---|---|---|---|
| H1 | La espera aumenta no-show | 16,0 % vs 32,7 %; `WaitingBin` creciente | barras por `WaitingBin` |
| H2 | Edad aporta señal propia | Adolescencia 26,6 %, Senior 15,5 % | heatmap `AgeGroup x WaitingBin` |
| H3 | Hay barrios de alto riesgo | `min_n=50`; 16 hotspots | top barrios fiables |

### 11.2.3.- Pregunta nueva para S04

```text
Pregunta:
  ¿barrio aporta informacion adicional a edad + espera?

Importancia:
  si es redundante, complica la operativa sin ganar decision.

Evidencia necesaria:
  grafico barrio vs espera mediana; despues, modelo con/sin barrio.
```

## 11.3.- Puente a S04

```text
S04 = de tablas a patrones visuales
  - cada grafico responde una pregunta EDA o una hipotesis PECO
  - 4 familias: Distribucion, Comparacion, Relacion, Composicion
  - producto final: 3 visualizaciones + 3 lecturas accionables
```

S02 nos dio evidencia tabular. S03 nos dio un **dataset preparado y reproducible** + tablas por segmento. S04 convierte esas tablas en **evidencia visual** que un stakeholder lea en 30 segundos.

## 11.4.- Cierre

Resultado: `prepare_basic(df)` -> `prepare(df)` -> `enrich_with_neighbourhood_rate(df)` -> `noshow_prepared.csv`.

Idea clave: **Data Preparation = decisiones trazables + dataset reproducible**.

Antes de S04: actualiza Canvas v3 con casilla 6 e hipotesis priorizadas.